# CVPR 2027 controlled editing pilot

Select an A100 GPU. This is a language-only action policy on synthetic templates, not a full vision benchmark. Three fixed epochs; effective batch size 12; strict and fence-normalized action scores; no test-based checkpoint selection.


In [ ]:
import subprocess, sys
subprocess.run(['nvidia-smi'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.51.3','peft==0.15.2','accelerate==1.6.0','scipy==1.15.3'],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','bitsandbytes'])


In [ ]:
from pathlib import Path
Path('/content/controlled_editing_data.py').write_text('"""Synthetic solver-linked edit-policy pilot, NOT a full scientific SVG benchmark.\n\nThe model receives a compact DOM index and emits one action. A deterministic\nexecutor owns geometry and labels. Train/test split by physical case; annuli are\nheld out. Natural-language templates are deliberately shared: this measures\ncase transfer, not human-instruction or language-template generalization.\n"""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport re\nimport xml.etree.ElementTree as ET\n\nimport numpy as np\nfrom scipy.sparse import diags, eye, kron\nfrom scipy.sparse.linalg import spsolve\n\nSYSTEM = \'\'\'Return exactly one JSON object for an engineering SVG edit.\nAllowed schemas:\n{"action":"style","target":"contour-id","attribute":"stroke","value":"#rrggbb"}\n{"action":"style","target":"contour-id","attribute":"stroke-width","value":number}\n{"action":"move_label","target":"label-id","x":number,"y":number}\n{"action":"recompute","parameter":"left_temperature","value":number}\n{"action":"reject","reason":"numerical_claim"}\n{"action":"reject","reason":"missing_required_contour"}\nPreserve every required contour\'s numerical value, geometry and visibility.\nChanging a physical boundary requires a new solve. Relabeling a contour to a\ndifferent physical value without changing geometry is forbidden. Labels can move\nonly inside the right annotation panel. Styling does not require a new solve.\nHex colors are copied verbatim. Do not include explanations or markdown.\'\'\'\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\',\', \':\')).encode()).hexdigest()\n\n\ndef solve_plate(width, height, left, right, n=25):\n    """-Laplacian T=0, left/right prescribed; insulated top/bottom.\n\n    FD on interior x nodes and all y nodes, zero-flux graph boundary on y.\n    Exact solution is linear in x; check it independently on every instance.\n    This simple manufactured control is not a challenging PDE benchmark.\n    """\n    x, y = np.linspace(0, width, n), np.linspace(0, height, n)\n    nx, ny = n-2, n\n    dx, dy = x[1], y[1]\n    ax = diags([-np.ones(nx-1), 2*np.ones(nx), -np.ones(nx-1)], [-1,0,1])/dx**2\n    diagonal = 2*np.ones(ny); diagonal[[0,-1]] = 1\n    ay = diags([-np.ones(ny-1), diagonal, -np.ones(ny-1)], [-1,0,1])/dy**2\n    matrix = kron(eye(ny), ax)+kron(ay, eye(nx))\n    rhs = np.zeros((ny,nx)); rhs[:,0] += left/dx**2; rhs[:,-1] += right/dx**2\n    interior = spsolve(matrix.tocsr(), rhs.ravel())\n    field = np.column_stack([np.full(ny,left), interior.reshape(ny,nx), np.full(ny,right)])\n    exact = left+(right-left)*x/width\n    error = float(np.max(np.abs(field-exact)))\n    assert error < 1e-8\n    residual = float(np.max(np.abs(matrix@interior-rhs.ravel())) / max(np.max(np.abs(rhs)),1))\n    return x,y,field,{\'method\':\'finite-difference sparse solve\', \'grid\':[n,n],\n                      \'max_error_against_exact_K\':error, \'relative_linear_residual\':residual}\n\n\ndef make_document(case_id, rng, annulus=False):\n    width,height = round(rng.uniform(.7,2.5),4),round(rng.uniform(.8,2.0),4)\n    cold,hot = rng.randint(270,310),rng.randint(340,410)\n    field = {\'quantity\':\'temperature\',\'unit\':\'K\',\'left_temperature\':hot,\n             \'right_temperature\':cold,\'width_m\':width,\'height_m\':height,\n             \'family\':\'annulus\' if annulus else \'insulated_rectangle\'}\n    levels = [round(cold+(hot-cold)*f,3) for f in (.25,.5,.75)]\n    if annulus:\n        # Analytic radial Laplace solution between concentric circles, hot inside.\n        field.update(inner_radius_m=.25*width,outer_radius_m=width)\n        reference = {\'method\':\'analytic radial Laplace\', \'formula\':\'cold+(hot-cold)*log(R/r)/log(R/a)\'}\n    else:\n        _,_,_,reference = solve_plate(width,height,hot,cold)\n    record = {\'case_id\':case_id,\'physical_problem\':field,\'reference\':reference}\n    solution_id = digest(record)\n    root = ET.Element(\'svg\',xmlns=\'http://www.w3.org/2000/svg\',viewBox=\'0 0 640 460\')\n    group = ET.SubElement(root,\'g\',id=\'field\', **{\'data-solution-id\':solution_id})\n    contours,labels = [],[]\n    for level in levels:\n        cid = \'c\'+str(rng.randrange(1000,9999)); lid = \'l\'+cid[1:]\n        if annulus:\n            radius = 180*math.exp(-(level-cold)/(hot-cold)*math.log(4))\n            ET.SubElement(group,\'circle\',id=cid,cx=\'220\',cy=\'230\',r=f\'{radius:.8f}\',fill=\'none\',stroke=\'#334155\',\n                          **{\'stroke-width\':\'1.5\',\'data-level\':str(level)})\n        else:\n            xpos=40+360*(hot-level)/(hot-cold)\n            ET.SubElement(group,\'path\',id=cid,d=f\'M{xpos:.8f} 40 L{xpos:.8f} 420\',fill=\'none\',stroke=\'#334155\',\n                          **{\'stroke-width\':\'1.5\',\'data-level\':str(level)})\n        contours.append({\'id\':cid,\'level\':level,\'unit\':\'K\',\'required\':True})\n        label=ET.SubElement(root,\'text\',id=lid,x=\'470\',y=str(100+80*len(labels)), **{\'data-contour\':cid})\n        label.text=f\'{level:g} K\'\n        labels.append({\'id\':lid,\'contour\':cid,\'text\':label.text})\n    return {\'solution_id\':solution_id,\'record\':record,\'contours\':contours,\'labels\':labels,\n            \'svg\':ET.tostring(root,encoding=\'unicode\')}\n\n\ndef execute(document, action):\n    """Whitelist executor for this generated schema; no arbitrary SVG accepted.\n\n    Does not interpret natural language. A policy model can choose the wrong\n    permitted action; semantic correctness is measured separately against target.\n    Recompute only returns a pending request, never a newly verified solution.\n    """\n    if not isinstance(action,dict):\n        raise ValueError(\'action must be object\')\n    original = copy.deepcopy(document)\n    root=ET.fromstring(document[\'svg\']); nodes={e.get(\'id\'):e for e in root.iter() if e.get(\'id\')}\n    name=action.get(\'action\')\n    def keys(wanted):\n        if set(action) != set(wanted.split()): raise ValueError(\'invalid action schema\')\n    def numeric(v):\n        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(v): raise ValueError(\'finite number required\')\n    if name==\'style\':\n        keys(\'action target attribute value\')\n        if action[\'target\'] not in {c[\'id\'] for c in document[\'contours\']}: raise ValueError(\'unknown contour\')\n        attr,val=action[\'attribute\'],action[\'value\']\n        if attr==\'stroke\':\n            if not isinstance(val,str) or not re.fullmatch(r\'#[0-9a-fA-F]{6}\',val): raise ValueError(\'hex color required\')\n            # White/near-white cannot be allowed to make required curves disappear.\n            if min(int(val[i:i+2],16) for i in (1,3,5))>220: raise ValueError(\'insufficient contrast\')\n        elif attr==\'stroke-width\':\n            numeric(val)\n            if not .5<=val<=4: raise ValueError(\'width outside visible bounds\')\n        else: raise ValueError(\'protected property\')\n        nodes[action[\'target\']].set(attr,str(val))\n    elif name==\'move_label\':\n        keys(\'action target x y\')\n        if action[\'target\'] not in {l[\'id\'] for l in document[\'labels\']}: raise ValueError(\'unknown label\')\n        numeric(action[\'x\']);numeric(action[\'y\'])\n        if not (460<=action[\'x\']<=540 and 30<=action[\'y\']<=430): raise ValueError(\'label outside annotation panel\')\n        for attr in (\'x\',\'y\'): nodes[action[\'target\']].set(attr,str(action[attr]))\n    elif name==\'recompute\':\n        keys(\'action parameter value\');numeric(action[\'value\'])\n        if action[\'parameter\']!=\'left_temperature\' or not 0<action[\'value\']<2000: raise ValueError(\'unsupported physical input\')\n        return {\'status\':\'requires_solver\',\'document\':original,\'request\':action}\n    elif name==\'reject\':\n        keys(\'action reason\')\n        if action[\'reason\'] not in (\'numerical_claim\',\'missing_required_contour\'): raise ValueError(\'unknown reason\')\n        return {\'status\':\'rejected\',\'document\':original}\n    else: raise ValueError(\'unknown action\')\n    original[\'svg\']=ET.tostring(root,encoding=\'unicode\')\n    return {\'status\':\'edited\',\'document\':original}\n\n\ndef examples(doc,rng,split):\n    c=rng.choice(doc[\'contours\']); l=next(l for l in doc[\'labels\'] if l[\'contour\']==c[\'id\'])\n    color=rng.choice([\'#2563eb\',\'#dc2626\',\'#059669\',\'#7c3aed\',\'#d97706\',\'#0e7490\'])\n    width=rng.choice([.75,1,2,2.5,3]); x=rng.randint(465,535);y=rng.randint(35,420)\n    newtemp=rng.randint(330,450)\n    tasks=[\n        (rng.choice([f"Set contour {c[\'id\']}\'s stroke to {color}.",f"Use {color} for the line color of {c[\'id\']}."]),\n         {\'action\':\'style\',\'target\':c[\'id\'],\'attribute\':\'stroke\',\'value\':color}),\n        (rng.choice([f"Give {c[\'id\']} a stroke width of {width}.",f"Change line thickness of contour {c[\'id\']} to {width} SVG units."]),\n         {\'action\':\'style\',\'target\':c[\'id\'],\'attribute\':\'stroke-width\',\'value\':width}),\n        (rng.choice([f"Move label {l[\'id\']} to x={x}, y={y} in the annotation panel.",f"Place the text {l[\'id\']} at ({x}, {y}); preserve its value and units."]),\n         {\'action\':\'move_label\',\'target\':l[\'id\'],\'x\':x,\'y\':y}),\n        (rng.choice([f"Change the hot boundary temperature to {newtemp} K and update the solution.",f"Solve the problem again with left_temperature={newtemp} K."]),\n         {\'action\':\'recompute\',\'parameter\':\'left_temperature\',\'value\':newtemp}),\n        (rng.choice([f"Keep the geometry but label the {c[\'level\']} K curve as {c[\'level\']+20:g} K.",f"Replace the value on {l[\'id\']} with {c[\'level\']+20:g} K without solving again."]),\n         {\'action\':\'reject\',\'reason\':\'numerical_claim\'}),\n        (rng.choice([f"Hide required contour {c[\'id\']} so the plot looks cleaner.",f"Remove the required curve {c[\'id\']} from the displayed drawing."]),\n         {\'action\':\'reject\',\'reason\':\'missing_required_contour\'})]\n    index={\'physical_problem\':doc[\'record\'][\'physical_problem\'],\'contours\':doc[\'contours\'],\'labels\':doc[\'labels\'],\n           \'annotation_panel\':{\'x\':[460,540],\'y\':[30,430]}}\n    rows=[]\n    for i,(instruction,target) in enumerate(tasks):\n        # An executor validity check is not a general physics correctness proof.\n        execute(doc,target)\n        rows.append({\'id\':f"{doc[\'record\'][\'case_id\']}-{i}",\'case_id\':doc[\'record\'][\'case_id\'],\'split\':split,\n                     \'instruction\':instruction,\'input\':json.dumps(index,separators=(\',\',\':\'))+\'\\nRequest: \'+instruction,\n                     \'target\':target,\'document\':doc})\n    return rows\n\n\ndef build(output,seed=2027,counts=(60,10,10,10)):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True);rng=random.Random(seed)\n    manifests={};all_cases=set()\n    for split,count in zip((\'train\',\'validation\',\'test\',\'ood\'),counts):\n        rows=[]\n        for i in range(count):\n            case_id=f\'{split}-{i:04d}\';assert case_id not in all_cases;all_cases.add(case_id)\n            doc=make_document(case_id,rng,annulus=split==\'ood\');rows.extend(examples(doc,rng,split))\n        rng.shuffle(rows)\n        content=\'\'.join(json.dumps(r,separators=(\',\',\':\'))+\'\\n\' for r in rows)\n        (output/f\'{split}.jsonl\').write_text(content)\n        manifests[split]={\'cases\':count,\'examples\':len(rows),\'sha256\':hashlib.sha256(content.encode()).hexdigest()}\n    manifest={\'version\':\'controlled-editing-pilot-v1\',\'seed\':seed,\'splits\':manifests,\n              \'limitations\':[\'synthetic shared instruction templates\',\'compact DOM index, not vision input\',\n                             \'simple manufactured heat equations\',\'recompute is routing only\',\n                             \'not evidence of CVPR-level novelty or complete SVG verification\']}\n    (output/\'manifest.json\').write_text(json.dumps(manifest,indent=2));return manifest\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--output\',default=\'runs/cvpr2027/controlled-editing-data\')\n    a=p.parse_args();print(json.dumps(build(a.output),indent=2))\n')
Path('/content/train_controlled_editing.py').write_text('"""LoRA pilot with completion-only loss, fixed epochs and before/after evaluation.\n\nRun on a CUDA GPU. Checkpoints, raw generations, package versions and hashes are\nsaved locally; Colab exports the resulting archive. No cloud logging or API keys.\n"""\nfrom __future__ import annotations\nimport argparse\nfrom collections import defaultdict\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nfrom controlled_editing_data import SYSTEM, build, execute\n\n\ndef load_rows(path):\n    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]\n\n\ndef score_action(row,text):\n    result={\'json_valid\':False,\'action_correct\':False,\'exact_action\':False,\'executor_accepts\':False,\n            \'unsafe_edit\':False,\'status\':None,\'fence_normalized_exact_action\':False}\n    # Report formatting sensitivity independently; never extract one convenient\n    # object from a multiple-action/truncated response.\n    clean=text.strip()\n    fenced=re.fullmatch(r\'```(?:json)?\\s*\\n?(.*?)\\n?```\',clean,re.DOTALL|re.IGNORECASE)\n    if fenced:clean=fenced[1].strip()\n    try:result[\'fence_normalized_exact_action\']=json.loads(clean)==row[\'target\']\n    except (ValueError,TypeError):pass\n    try:\n        action=json.loads(text)\n        if not isinstance(action,dict):return result\n        result[\'json_valid\']=True\n        result[\'action_correct\']=action.get(\'action\')==row[\'target\'][\'action\']\n        result[\'exact_action\']=action==row[\'target\']\n        outcome=execute(row[\'document\'],action)\n        result[\'executor_accepts\']=True;result[\'status\']=outcome[\'status\']\n        # Semantic misuse of a permitted edit is still wrong; executor protection\n        # alone cannot determine whether an instruction was fulfilled honestly.\n        result[\'unsafe_edit\']=outcome[\'status\']==\'edited\' and row[\'target\'][\'action\'] in (\'reject\',\'recompute\')\n    except (ValueError,TypeError,KeyError):pass\n    return result\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--output\',default=\'/content/svg-editing-pilot\')\n    p.add_argument(\'--seed\',type=int,default=17);p.add_argument(\'--epochs\',type=int,default=3)\n    p.add_argument(\'--resume\',action=\'store_true\');p.add_argument(\'--model\',default=\'Qwen/Qwen2.5-Coder-1.5B-Instruct\')\n    a=p.parse_args();out=Path(a.output);out.mkdir(parents=True,exist_ok=True)\n    import torch\n    from huggingface_hub import model_info\n    from transformers import (AutoModelForCausalLM,AutoTokenizer,\n                              Trainer,TrainingArguments,set_seed)\n    from peft import LoraConfig,get_peft_model\n    if not torch.cuda.is_available():raise RuntimeError(\'CUDA GPU required; CPU fallback disabled\')\n    set_seed(a.seed)\n    data_dir=out/\'data\'\n    if not (data_dir/\'manifest.json\').exists():build(data_dir)\n    data={s:load_rows(data_dir/f\'{s}.jsonl\') for s in (\'train\',\'validation\',\'test\',\'ood\')}\n    case_sets=[{r[\'case_id\'] for r in rows} for rows in data.values()]\n    assert sum(map(len,case_sets))==len(set.union(*case_sets)), \'physical-case leakage\'\n    manifest_path=out/\'run_manifest.json\'\n    if a.resume and manifest_path.exists():\n        previous=json.loads(manifest_path.read_text())\n        if previous[\'seed\']!=a.seed or previous[\'model\']!=a.model or previous[\'epochs\']!=a.epochs:\n            raise ValueError(\'resume configuration mismatch\')\n        revision=previous[\'model_revision\']\n    else:\n        if manifest_path.exists():raise ValueError(\'existing run: use --resume or a new output directory\')\n        revision=model_info(a.model).sha\n    manifest={\'model\':a.model,\'model_revision\':revision,\'seed\':a.seed,\'epochs\':a.epochs,\n              \'gpu\':torch.cuda.get_device_name(0),\'torch\':torch.__version__,\'status\':\'started\',\n              \'micro_batch_size\':4,\'gradient_accumulation_steps\':3,\'effective_batch_size\':12,\n              \'data_manifest\':json.loads((data_dir/\'manifest.json\').read_text()),\n              \'training_script_sha256\':hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n              \'data_script_sha256\':hashlib.sha256(Path(__file__).with_name(\'controlled_editing_data.py\').read_bytes()).hexdigest(),\n              \'selection\':\'fixed final epoch, no test-based model selection\'}\n    manifest_path.write_text(json.dumps(manifest,indent=2))\n    (out/\'packages.txt\').write_text(subprocess.check_output([sys.executable,\'-m\',\'pip\',\'freeze\'],text=True))\n    tokenizer=AutoTokenizer.from_pretrained(a.model,revision=revision)\n    tokenizer.pad_token=tokenizer.eos_token\n    model=AutoModelForCausalLM.from_pretrained(a.model,revision=revision,\n                                              device_map={\'\':0},torch_dtype=torch.float16)\n    model.enable_input_require_grads()\n    model=get_peft_model(model,LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,\n                          target_modules=[\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\'],task_type=\'CAUSAL_LM\'))\n    model.print_trainable_parameters()\n\n    def prompt(row):\n        return tokenizer.apply_chat_template([{\'role\':\'system\',\'content\':SYSTEM},\n                    {\'role\':\'user\',\'content\':row[\'input\']}],tokenize=False,add_generation_prompt=True)\n\n    def evaluate(label):\n        file=out/f\'{label}_generations.jsonl\';existing={}\n        if file.exists():\n            for r in load_rows(file):existing[r[\'id\']]=r\n        model.eval();tokenizer.padding_side=\'left\';model.config.use_cache=True\n        # Raw generation before training is saved; base uses disabled LoRA adapters.\n        for split in (\'test\',\'ood\'):\n            todo=[r for r in data[split] if r[\'id\'] not in existing]\n            for start in range(0,len(todo),4):\n                rows=todo[start:start+4];inputs=tokenizer([prompt(r) for r in rows],return_tensors=\'pt\',padding=True).to(\'cuda\')\n                if inputs.input_ids.shape[1]>1536:raise ValueError(\'evaluation prompt too long; never silently truncate\')\n                t0=time.perf_counter()\n                with torch.inference_mode():\n                    generated=model.generate(**inputs,max_new_tokens=128,do_sample=False,\n                                             pad_token_id=tokenizer.pad_token_id)\n                answers=tokenizer.batch_decode(generated[:,inputs.input_ids.shape[1]:],skip_special_tokens=True)\n                with file.open(\'a\') as handle:\n                    for row,text in zip(rows,answers):\n                        result={\'id\':row[\'id\'],\'case_id\':row[\'case_id\'],\'split\':split,\'target\':row[\'target\'],\n                                \'prediction\':text,\'metrics\':score_action(row,text),\n                                \'batch_seconds\':time.perf_counter()-t0}\n                        handle.write(json.dumps(result)+\'\\n\');handle.flush();existing[row[\'id\']]=result\n                print(label,split,min(start+4,len(todo)),\'/\',len(todo),flush=True)\n        summary={}\n        for split in (\'test\',\'ood\'):\n            rows=[existing[r[\'id\']] for r in data[split]]\n            summary[split]={\'n\':len(rows)}\n            for metric in (\'json_valid\',\'action_correct\',\'exact_action\',\'executor_accepts\',\'unsafe_edit\',\'fence_normalized_exact_action\'):\n                summary[split][metric]=sum(r[\'metrics\'][metric] for r in rows)/len(rows)\n            groups=defaultdict(list)\n            for r in rows:\n                kind=r[\'target\'][\'action\']\n                if kind==\'style\':kind+=\':\'+r[\'target\'][\'attribute\']\n                if kind==\'reject\':kind+=\':\'+r[\'target\'][\'reason\']\n                groups[kind].append(r[\'metrics\'][\'exact_action\'])\n            summary[split][\'exact_by_task\']={k:{\'n\':len(v),\'rate\':sum(v)/len(v)} for k,v in groups.items()}\n        (out/f\'{label}_metrics.json\').write_text(json.dumps(summary,indent=2))\n        print(label,json.dumps(summary),flush=True)\n\n    with model.disable_adapter():evaluate(\'base\')\n    tokenizer.padding_side=\'right\';model.config.use_cache=False\n    def tokenize(rows):\n        examples=[]\n        for row in rows:\n            prefix=tokenizer(prompt(row),add_special_tokens=False)[\'input_ids\']\n            answer=tokenizer(json.dumps(row[\'target\'],separators=(\',\',\':\'))+tokenizer.eos_token,\n                             add_special_tokens=False)[\'input_ids\']\n            ids=prefix+answer\n            if len(ids)>1536:raise ValueError(\'training example too long; no silent truncation\')\n            examples.append({\'input_ids\':ids,\'labels\':[-100]*len(prefix)+answer})\n        return examples\n\n    def collate(rows):\n        length=max(len(r[\'input_ids\']) for r in rows)\n        return {k:torch.tensor([r[k]+[tokenizer.pad_token_id if k==\'input_ids\' else -100]*(length-len(r[k]))\n                               for r in rows]) for k in (\'input_ids\',\'labels\')} | {\n            \'attention_mask\':torch.tensor([[1]*len(r[\'input_ids\'])+[0]*(length-len(r[\'input_ids\'])) for r in rows])}\n    args=TrainingArguments(output_dir=str(out/\'checkpoints\'),num_train_epochs=a.epochs,\n         per_device_train_batch_size=4,gradient_accumulation_steps=3,per_device_eval_batch_size=4,\n         learning_rate=2e-4,lr_scheduler_type=\'cosine\',warmup_ratio=.05,weight_decay=.01,\n         fp16=True,gradient_checkpointing=True,gradient_checkpointing_kwargs={\'use_reentrant\':False},\n         logging_steps=5,eval_strategy=\'epoch\',save_strategy=\'epoch\',save_total_limit=2,\n         report_to=\'none\',seed=a.seed,data_seed=a.seed,remove_unused_columns=False,label_names=[\'labels\'],\n         optim=\'adamw_torch\',dataloader_num_workers=0)\n    trainer=Trainer(model=model,args=args,train_dataset=tokenize(data[\'train\']),\n                    eval_dataset=tokenize(data[\'validation\']),data_collator=collate)\n    checkpoints=sorted((out/\'checkpoints\').glob(\'checkpoint-*\'),key=lambda p:int(p.name.split(\'-\')[-1]))\n    t0=time.perf_counter();result=trainer.train(resume_from_checkpoint=str(checkpoints[-1]) if a.resume and checkpoints else None)\n    model.save_pretrained(out/\'adapter\');tokenizer.save_pretrained(out/\'adapter\')\n    (out/\'training_metrics.json\').write_text(json.dumps(result.metrics,indent=2))\n    trainer.state.save_to_json(str(out/\'trainer_state.json\'))\n    training_s=time.perf_counter()-t0\n    model.gradient_checkpointing_disable();evaluate(\'sft\')\n    manifest.update(status=\'completed\',training_seconds=training_s,global_steps=trainer.state.global_step,\n                    max_gpu_memory_allocated_bytes=torch.cuda.max_memory_allocated())\n    manifest_path.write_text(json.dumps(manifest,indent=2))\n    print(\'RUN COMPLETE\',str(out),flush=True)\n\n\nif __name__==\'__main__\':main()\n')


In [ ]:
import subprocess, sys, shutil
from pathlib import Path
from google.colab import files
root = Path('/content/cvpr2027-a100'); root.mkdir(exist_ok=True)
for seed in (17, 29, 41):
    run = root / f'seed-{seed}'
    with (root/f'seed-{seed}.log').open('a') as log:
        process = subprocess.Popen([sys.executable, '-u', '/content/train_controlled_editing.py', '--output', str(run), '--epochs', '3', '--seed', str(seed), '--resume'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = process.wait()
    assert code == 0, f'Seed {seed} failed: {code}'
    print('COMPLETED SEED', seed, flush=True)
shutil.make_archive('/content/cvpr2027-a100', 'zip', root)
print('ALL THREE SEEDS COMPLETE; downloading')
files.download('/content/cvpr2027-a100.zip')


In [ ]:
# Retry download only if the automatic download did not complete.
from google.colab import files
files.download('/content/cvpr2027-a100.zip')


In [ ]:
from pathlib import Path
Path('/content/check_adapter_reload.py').write_text('"""CUDA smoke check: reload each saved adapter and run six held-out actions."""\nimport argparse\nimport gc\nimport json\nfrom pathlib import Path\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--runs\', type=Path, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    args = p.parse_args()\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from controlled_editing_data import SYSTEM\n    from train_controlled_editing import load_rows, score_action\n    results = []\n    for seed in (17, 29, 41):\n        run = args.runs / f\'seed-{seed}\'\n        manifest = json.loads((run / \'run_manifest.json\').read_text())\n        assert manifest[\'status\'] == \'completed\'\n        tokenizer = AutoTokenizer.from_pretrained(run / \'adapter\')\n        base = AutoModelForCausalLM.from_pretrained(\n            manifest[\'model\'], revision=manifest[\'model_revision\'],\n            device_map={\'\': 0}, torch_dtype=torch.float16)\n        model = PeftModel.from_pretrained(base, run / \'adapter\').eval()\n        selected = {}\n        for row in load_rows(run / \'data\' / \'test.jsonl\'):\n            target = row[\'target\']\n            key = (target[\'action\'], target.get(\'attribute\'), target.get(\'reason\'))\n            selected.setdefault(key, row)\n        assert len(selected) == 6\n        for row in selected.values():\n            prompt = tokenizer.apply_chat_template([\n                {\'role\': \'system\', \'content\': SYSTEM},\n                {\'role\': \'user\', \'content\': row[\'input\']}],\n                tokenize=False, add_generation_prompt=True)\n            inputs = tokenizer(prompt, return_tensors=\'pt\').to(\'cuda\')\n            with torch.inference_mode():\n                generated = model.generate(**inputs, max_new_tokens=128,\n                    do_sample=False, pad_token_id=tokenizer.eos_token_id)\n            text = tokenizer.decode(generated[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)\n            score = score_action(row, text)\n            results.append({\'seed\': seed, \'id\': row[\'id\'], \'prediction\': text,\n                            \'target\': row[\'target\'], \'metrics\': score})\n            print(seed, row[\'id\'], score[\'exact_action\'], flush=True)\n        del model, base, inputs, generated\n        gc.collect(); torch.cuda.empty_cache()\n    report = {\'scope\': \'saved-adapter reload smoke check; six actions per seed\',\n              \'n\': len(results), \'exact\': sum(r[\'metrics\'][\'exact_action\'] for r in results),\n              \'results\': results}\n    args.output.write_text(json.dumps(report, indent=2) + \'\\n\')\n    assert report[\'exact\'] == report[\'n\'] == 18, \'Reload predictions require inspection\'\n    print(\'ADAPTER RELOAD VERIFIED\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
import subprocess, sys
subprocess.run([sys.executable, '/content/check_adapter_reload.py', '--runs', '/content/cvpr2027-a100', '--output', '/content/adapter-reload-check.json'], check=True)
from google.colab import files
files.download('/content/adapter-reload-check.json')
